
# Task 8 – Building a Collaborative Filtering Recommendation Model

## Objective

The objective of this task is to develop a User-Based Collaborative Filtering Recommendation Model that recommends TED Talks based on user preferences and interaction patterns.

In this task, we use the simulated user interaction dataset created in Task 7. We create a User-Item Interaction Matrix, calculate cosine similarity between users, identify similar users, and generate personalized TED Talk recommendations.

The recommendation model identifies users with similar interests and recommends TED Talks based on their ratings and interaction history.


In [20]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

In [21]:
interaction_path = Path(r"C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\interaction\user_interactions.csv")

interactions = pd.read_csv(interaction_path)

print("Dataset Shape:", interactions.shape)

display(interactions.head())

Dataset Shape: (597, 7)


,user_id,talk_id,rating,liked,disliked,watched,saved
0,U001,T044,5,True,False,True,True
1,U001,T041,4,True,False,True,False
2,U001,T047,4,True,False,True,False
3,U001,T013,3,False,False,True,False
4,U001,T025,4,True,False,True,True


In [22]:
print("\nNumber of users:", interactions["user_id"].nunique())
print("Number of TED Talks:", interactions["talk_id"].nunique())

print("\nColumns:")
print(interactions.columns.tolist())


Number of users: 30
Number of TED Talks: 50

Columns:
['user_id', 'talk_id', 'rating', 'liked', 'disliked', 'watched', 'saved']


In [23]:
# Create the User-Item Interaction Matrix
user_item_matrix = interactions.pivot(
    index="user_id",
    columns="talk_id",
    values="rating"
)

# Display the matrix
display(user_item_matrix)

talk_id,T001,T002,T003,T004,T005,T006,T007,T008,T009,T010,...,T041,T042,T043,T044,T045,T046,T047,T048,T049,T050
user_id,,,,,,,,,,,,,,,,,,,,,
U001,NaN,NaN,NaN,2.0,1.0,NaN,2.0,NaN,5.0,NaN,...,4.0,NaN,NaN,5.0,NaN,NaN,4.0,4.0,3.0,2.0
U002,NaN,4.0,NaN,1.0,NaN,3.0,NaN,NaN,NaN,5.0,...,NaN,NaN,5.0,5.0,1.0,4.0,1.0,2.0,3.0,NaN
U003,3.0,5.0,NaN,NaN,NaN,NaN,NaN,5.0,NaN,4.0,...,4.0,3.0,NaN,NaN,3.0,3.0,NaN,1.0,NaN,NaN
U004,NaN,4.0,NaN,NaN,NaN,NaN,5.0,NaN,4.0,NaN,...,4.0,NaN,2.0,4.0,5.0,4.0,NaN,NaN,NaN,NaN
U005,NaN,4.0,NaN,2.0,NaN,5.0,5.0,4.0,NaN,1.0,...,NaN,NaN,3.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN
U006,1.0,2.0,NaN,NaN,NaN,2.0,NaN,5.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3.0,5.0,4.0,5.0,NaN
U007,NaN,4.0,NaN,1.0,5.0,NaN,3.0,NaN,2.0,NaN,...,NaN,1.0,5.0,NaN,1.0,NaN,4.0,NaN,NaN,NaN
U008,NaN,4.0,NaN,1.0,NaN,NaN,5.0,2.0,5.0,2.0,...,1.0,NaN,2.0,3.0,NaN,NaN,NaN,NaN,3.0,NaN
U009,NaN,4.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN,2.0,...,3.0,4.0,NaN,NaN,5.0,4.0,1.0,NaN,2.0,3.0


In [24]:
# Replace missing values with zero for similarity calculation
user_item_filled = user_item_matrix.fillna(0)

# Display the filled matrix
display(user_item_filled.head())

talk_id,T001,T002,T003,T004,T005,T006,T007,T008,T009,T010,...,T041,T042,T043,T044,T045,T046,T047,T048,T049,T050
user_id,,,,,,,,,,,,,,,,,,,,,
U001,0.0,0.0,0.0,2.0,1.0,0.0,2.0,0.0,5.0,0.0,...,4.0,0.0,0.0,5.0,0.0,0.0,4.0,4.0,3.0,2.0
U002,0.0,4.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,5.0,...,0.0,0.0,5.0,5.0,1.0,4.0,1.0,2.0,3.0,0.0
U003,3.0,5.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,4.0,...,4.0,3.0,0.0,0.0,3.0,3.0,0.0,1.0,0.0,0.0
U004,0.0,4.0,0.0,0.0,0.0,0.0,5.0,0.0,4.0,0.0,...,4.0,0.0,2.0,4.0,5.0,4.0,0.0,0.0,0.0,0.0
U005,0.0,4.0,0.0,2.0,0.0,5.0,5.0,4.0,0.0,1.0,...,0.0,0.0,3.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0


In [25]:
# Calculate cosine similarity between users
similarity_matrix = cosine_similarity(user_item_filled)

# Convert the similarity matrix into a DataFrame
user_similarity_df = pd.DataFrame(
    similarity_matrix,
    index=user_item_filled.index,
    columns=user_item_filled.index
)

# Display user similarity scores
display(user_similarity_df.head())

user_id,U001,U002,U003,U004,U005,U006,U007,U008,U009,U010,...,U021,U022,U023,U024,U025,U026,U027,U028,U029,U030
user_id,,,,,,,,,,,,,,,,,,,,,
U001,1.000000,0.369529,0.272077,0.449901,0.269700,0.303239,0.390038,0.415316,0.264102,0.430014,...,0.235214,0.425797,0.428161,0.322332,0.380534,0.527643,0.389917,0.358817,0.294727,0.512005
U002,0.369529,1.000000,0.360317,0.492126,0.460882,0.246183,0.358870,0.312840,0.378370,0.317501,...,0.466338,0.268947,0.483859,0.363602,0.487616,0.299439,0.549260,0.403620,0.449415,0.410553
U003,0.272077,0.360317,1.000000,0.455405,0.377742,0.247674,0.177918,0.337301,0.517126,0.316411,...,0.444132,0.098643,0.356304,0.272942,0.380385,0.284201,0.304016,0.307832,0.289103,0.214131
U004,0.449901,0.492126,0.455405,1.000000,0.332451,0.147497,0.406218,0.493944,0.432055,0.484892,...,0.327289,0.313276,0.291074,0.100478,0.225997,0.448513,0.390774,0.396266,0.321178,0.361056
U005,0.269700,0.460882,0.377742,0.332451,1.000000,0.247865,0.300060,0.340916,0.331655,0.214158,...,0.309524,0.167384,0.074112,0.318123,0.371870,0.260105,0.224975,0.268926,0.320388,0.245424


In [26]:
# Select a user
selected_user = "U001"

# Get similarity scores for the selected user
similar_users = user_similarity_df[selected_user].drop(
    selected_user
)

# Sort users by similarity
similar_users = similar_users.sort_values(
    ascending=False
)

print("Users most similar to", selected_user)
display(similar_users.head(10))

Users most similar to U001


user_id
U026    0.527643
U030    0.512005
U012    0.486612
U014    0.463524
U004    0.449901
U010    0.430014
U023    0.428161
U022    0.425797
U008    0.415316
U017    0.414548
Name: U001, dtype: float64

In [27]:
# Load the cleaned TED Talks dataset
talks = pd.read_csv(
    r"C:\Users\vramy\OneDrive\Desktop\samba-shiva-reddy\ted-talks-recommender\data\processed\ted_talks_cleaned.csv"
)

# Reset index and select the first 50 talks
talks = talks.reset_index(drop=True).head(50).copy()

# Generate the same Talk IDs as Notebook 7
talks["talk_id"] = [
    f"T{i+1:03d}" for i in range(len(talks))
]

# Create the talk catalog
talk_catalog = talks[
    ["talk_id", "title", "main_speaker"]
]

display(talk_catalog.head())

,talk_id,title,main_speaker
0,T001,do schools kill creativity?,Ken Robinson
1,T002,averting the climate crisis,Al Gore
2,T003,simplicity sells,David Pogue
3,T004,greening the ghetto,Majora Carter
4,T005,the best stats you've ever seen,Hans Rosling


In [28]:
def recommend_for_user(user_id, top_n=5):

    # Check whether the user exists
    if user_id not in user_item_matrix.index:
        raise ValueError(f"User ID '{user_id}' does not exist.")

    # Check whether top_n is valid
    if not isinstance(top_n, int) or isinstance(top_n, bool) or top_n <= 0:
        raise ValueError("top_n must be a positive integer.")

    # Get similarity scores of other users
    similar_users = user_similarity_df[user_id].drop(user_id)

    # Sort users by similarity
    similar_users = similar_users.sort_values(ascending=False)

    # Find talks that the selected user has not rated
    unrated_talks = user_item_matrix.loc[user_id][
        user_item_matrix.loc[user_id].isna()
    ].index

    recommendations = []

    # Calculate predicted rating for each unrated talk
    for talk_id in unrated_talks:

        weighted_sum = 0
        similarity_sum = 0

        # Consider ratings from similar users
        for other_user, similarity in similar_users.items():

            rating = user_item_matrix.loc[other_user, talk_id]

            # Use only users who rated this talk and have positive similarity
            if pd.notna(rating) and similarity > 0:

                weighted_sum += similarity * rating
                similarity_sum += similarity

        # Calculate predicted rating
        if similarity_sum > 0:

            predicted_rating = weighted_sum / similarity_sum

            recommendations.append({
                "talk_id": talk_id,
                "predicted_rating": predicted_rating
            })

    # Convert recommendations into a DataFrame
    recommendations_df = pd.DataFrame(recommendations)

    # Handle cases where no recommendations are available
    if recommendations_df.empty:
        return recommendations_df

    # Sort by predicted rating in descending order
    recommendations_df = recommendations_df.sort_values(
        by="predicted_rating",
        ascending=False
    )

    # Return top N recommendations
    return recommendations_df.head(top_n).reset_index(drop=True)

In [29]:
# Generate recommendations for U001
recommendations = recommend_for_user("U001", top_n=5)

# Add TED Talk titles and speaker names
recommendations = recommendations.merge(
    talk_catalog,
    on="talk_id",
    how="left"
)

print("Recommended TED Talks for U001:")

display(
    recommendations[
        ["talk_id", "title", "main_speaker", "predicted_rating"]
    ]
)

Recommended TED Talks for U001:


,talk_id,title,main_speaker,predicted_rating
0,T046,how technology evolves,Kevin Kelly,3.767631
1,T026,chemical scum that dream of distant quasars,David Deutsch,3.529739
2,T028,the freakonomics of crack dealing,Steven Levitt,3.498522
3,T021,the birth of the open-source learning revolution,Richard Baraniuk,3.478991
4,T002,averting the climate crisis,Al Gore,3.475467


In [30]:
# Test recommendations for three users
test_users = ["U001", "U002", "U003"]

for user_id in test_users:

    print("\nRecommendations for", user_id)

    user_recommendations = recommend_for_user(
        user_id,
        top_n=5
    )

    # Add TED Talk titles
    user_recommendations = user_recommendations.merge(
        talk_catalog,
        on="talk_id",
        how="left"
    )

    display(
        user_recommendations[
            ["talk_id", "title", "main_speaker", "predicted_rating"]
        ]
    )


Recommendations for U001


,talk_id,title,main_speaker,predicted_rating
0,T046,how technology evolves,Kevin Kelly,3.767631
1,T026,chemical scum that dream of distant quasars,David Deutsch,3.529739
2,T028,the freakonomics of crack dealing,Steven Levitt,3.498522
3,T021,the birth of the open-source learning revolution,Richard Baraniuk,3.478991
4,T002,averting the climate crisis,Al Gore,3.475467



Recommendations for U002


,talk_id,title,main_speaker,predicted_rating
0,T028,the freakonomics of crack dealing,Steven Levitt,3.655093
1,T007,letting go of god,Julia Sweeney,3.564190
2,T033,a roadmap to end aging,Aubrey de Grey,3.343020
3,T009,let's teach religion -- all religion -- in sch...,Dan Dennett,3.297927
4,T050,the killer american diet that's sweeping the p...,Dean Ornish,3.275934



Recommendations for U003


,talk_id,title,main_speaker,predicted_rating
0,T016,an 11-year-old's magical violin,Sirena Huang,3.605345
1,T007,letting go of god,Julia Sweeney,3.603901
2,T029,"choice, happiness and spaghetti sauce",Malcolm Gladwell,3.591964
3,T026,chemical scum that dream of distant quasars,David Deutsch,3.569277
4,T028,the freakonomics of crack dealing,Steven Levitt,3.379334


In [31]:
for user_id in ["U001", "U002", "U003"]:

    recs = recommend_for_user(user_id, top_n=5)

    # Get talks already rated by this user
    already_rated = set(
        user_item_matrix.loc[user_id].dropna().index
    )

    # Get recommended Talk IDs
    recommended_ids = set(recs["talk_id"])

    # Check that already-rated talks are not recommended
    assert already_rated.isdisjoint(recommended_ids)

    # Check that recommendations are sorted
    assert recs["predicted_rating"].is_monotonic_decreasing

    # Check that recommended Talk IDs exist
    assert recommended_ids.issubset(set(talk_catalog["talk_id"]))

print("All recommendation validation checks passed!")

All recommendation validation checks passed!


In [32]:
try:
    recommend_for_user("U999", top_n=5)

except ValueError as error:
    print("Error:", error)

Error: User ID 'U999' does not exist.



# Review Questions and Answers

### 1. What is Collaborative Filtering?

Collaborative Filtering is a recommendation technique that suggests items to users based on the preferences and interests of other users.

In our TED Talks recommendation system, if two users have similar rating patterns, TED Talks liked by one user can be recommended to the other user.

### 2. What is the difference between Content-Based Filtering and Collaborative Filtering?

**Content-Based Filtering:**
- Recommends items based on their content and features.
- Uses information such as title, description, tags, and transcript.
- Finds TED Talks similar to the talk selected by the user.

**Collaborative Filtering:**
- Recommends items based on user interactions and preferences.
- Uses ratings and interaction history.
- Finds users with similar interests and recommends talks they liked.

### 3. What is a User-Item Interaction Matrix?

A User-Item Interaction Matrix is a table that represents interactions between users and items.

- Rows represent users.
- Columns represent items.
- Values represent ratings or interactions.

In our project, rows represent users, columns represent TED Talks, and values represent ratings.

### 4. Why do we calculate similarity between users?

We calculate similarity to identify users who have similar interests and preferences.

If two users have similar rating patterns, the system can use one user's ratings to recommend TED Talks to the other user.

### 5. What is Cosine Similarity?

Cosine Similarity is a mathematical technique used to measure the similarity between two vectors.

It calculates the cosine of the angle between the vectors.

Its general range is from -1 to 1.

- 1 indicates highly similar directions.
- 0 indicates no directional similarity.
- -1 indicates opposite directions.

In our project, ratings are non-negative, so similarity scores are generally between 0 and 1.

### 6. Why do we exclude already-rated TED Talks from recommendations?

We exclude already-rated TED Talks because the user has already interacted with those talks.

The recommendation system should suggest new or unrated TED Talks instead of repeatedly recommending items the user has already rated.

### 7. What is a Predicted Rating?

A Predicted Rating is an estimated rating that a user might give to a TED Talk.

It is calculated using the ratings given by similar users, weighted by their similarity scores.

A higher predicted rating indicates that the TED Talk may be more suitable for the selected user.

### 8. What is the Cold-Start Problem?

The Cold-Start Problem occurs when the recommendation system does not have enough information about a new user or a new item.

**New User:** The system cannot identify similar users because the new user has no interaction history.

**New Item:** The system cannot recommend a new TED Talk effectively if no users have rated it.

### 9. What are the limitations of Collaborative Filtering?

1. It requires sufficient user interaction data.
2. New users may not receive personalized recommendations.
3. New items may not have enough ratings.
4. Sparse interaction matrices can reduce recommendation quality.
5. Similarity scores may not always represent actual user interests.
6. Simulated data may not reflect real-world user preferences.

### 10. Why do we use Cosine Similarity in our project?

We use Cosine Similarity to compare users based on their rating patterns.

It helps identify users with similar interests. Their ratings are then used to calculate predicted ratings and recommend TED Talks.


# Observations

1. A User-Item Interaction Matrix was created using ratings from 30 simulated users and 50 TED Talks.
2. Cosine similarity was used to identify users with similar interests.
3. Similar users were identified based on their rating patterns.
4. TED Talks that a user had not rated were considered for recommendations.
5. Predicted ratings were calculated using the ratings of similar users.
6. The recommendation system returned TED Talks in descending order of predicted ratings.
7. Recommendations were tested for three different users: U001, U002, and U003.
8. The system successfully excluded talks that users had already rated.
9. Different users can receive different recommendations based on their interaction history.

# Limitations

1. The interaction dataset contains simulated data rather than real user activity.
2. New users without interaction history cannot receive personalized recommendations effectively.
3. New TED Talks without ratings may not be recommended.
4. A small dataset may not represent real-world user preferences.
5. Replacing missing ratings with zero for cosine similarity is a simplification because missing ratings do not represent actual zero ratings.

# Conclusion

Collaborative filtering was implemented using user-based cosine similarity.

The system identifies users with similar interests and recommends TED Talks based on their ratings.

The recommendation function was tested with multiple users and successfully generated personalized recommendations using the simulated interaction dataset.